# Generate Missing Embeddings

**Runs on Colab or local Jupyter.** Recommended on Colab (free GPU) for the two open-source models, since they need GPU inference to finish in reasonable time across the corpus. If a local GPU is available this runs there too; if not, it still runs on CPU, just slower. This is detected automatically below, no manual switch needed.

**Purpose of this notebook**

Before the paradigm benchmark (pgvector vs MongoDB vs Qdrant) can begin, we need to pick which embedding model to standardize on. That decision requires comparing candidate models on retrieval quality first. This notebook fills the embedding gaps needed for that model comparison:

| Model | abstracts | pdf_chunks |
|---|---|---|
| `text-embedding-3-small` | done (full) | done (capped, ~100 papers/category) |
| `text-embedding-3-large` | done (full) | **missing — generated here** |
| `BAAI/bge-small-en-v1.5` | **missing — generated here** | **missing — generated here** |
| `intfloat/e5-base-v2` | **missing — generated here** | **missing — generated here** |

**Matching the existing cap, not introducing a new one**: `text-embedding-3-small`'s pdf_chunks were generated capped at the first 100 papers per category (by `paper_id`, sorted for determinism), not the full PDF corpus, for cost control. For the benchmark comparison to be fair, every model needs to be evaluated on the *same* set of chunks. So this notebook applies that identical cap to all three remaining models' pdf_chunks generation, using the identical selection logic (same `cap_pdf_records_by_paper` function, same `MAX_PDFS_PER_CATEGORY`). Abstracts stay uncapped (full corpus) for every model, matching the existing pattern there too.

**Why the open-source models want a GPU**: they need to run local inference over the corpus. `text-embedding-3-large` is an API call so it needs no GPU either way; it's included here so all generation lives in one notebook and one run leaves everything the evaluation notebook needs, ready in one place.

**Why we don't touch `text-embedding-3-small`**: it's already complete for both abstracts and (capped) pdf_chunks, regenerating it would burn API budget for zero new information.

**Output convention**: every model writes to `data/embeddings/{model_name}/{abstracts|pdf_chunks}/{category}.jsonl`, one JSON object per line: `{"chunk_id": ..., "embedding": [...]}`. This exactly matches the existing `text-embedding-3-small` files, so the evaluation notebook's loader doesn't need per-model special-casing.

**Source text**: pulled from the HF dataset `Sakhiur/empirical-rag-paradigm-benchmark`, `abstracts/by_category/{category}.jsonl` and `pdf_chunks/{category}.jsonl`. This notebook only reads text from HF and writes new embeddings back to HF; it never touches the raw corpus files.

## Setup

Works on Colab or local Jupyter. `IN_COLAB` (detected below) controls the couple of places behavior has to differ: secrets handling and GPU runtime messaging. Package installation is the same command either way; pip is idempotent about already-satisfied requirements, so re-running this cell on a machine that already has these packages is harmless.

In [ ]:
# sentence-transformers covers both open-source models with one API.
# huggingface_hub for pulling source text and pushing finished jsonl files back.
# python-dotenv is optional (only used locally, see the secrets cell below); harmless
# to install even on Colab where it won't be used.
%pip install -q sentence-transformers openai huggingface_hub python-dotenv

In [ ]:
import os
import json
import time
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = running_in_colab()
print(f"Environment: {'Colab' if IN_COLAB else 'local Jupyter'}")

### API keys and HF token

On Colab, secrets come from the Colab Secrets panel (key icon in the left sidebar) — this avoids ever typing a key into a cell. Locally, the same two environment variables (`HF_TOKEN`, `OPENAI_API_KEY`) are expected to already be set in the shell that launched Jupyter, or in a `.env` file loaded by `python-dotenv`, matching the pattern in your existing question-generation notebook. Either path lands in the same place: both variables present in `os.environ` before the next cell runs.

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    # Local: try a .env file first (via python-dotenv), then fall back to whatever
    # is already in the shell environment. Never prompts or hardcodes a key here.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    missing = [k for k in ("HF_TOKEN", "OPENAI_API_KEY") if not os.environ.get(k)]
    if missing:
        raise RuntimeError(
            f"Missing required environment variable(s): {missing}. "
            "Set them in your shell (export HF_TOKEN=...) or in a .env file "
            "in the notebook's working directory before re-running this cell."
        )

from huggingface_hub import whoami, login

try:
    user_info = whoami()
    print(f"Hugging Face authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated with Hugging Face yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## Configuration

Central place to control which models and content types get generated this run, and the cap that keeps this run cost- and time-bounded. Kept as a single cell so a partial run (e.g. only the open-source models, or only pdf_chunks) is a one-line edit rather than hunting through the notebook.

In [ ]:
HF_REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
HF_REPO_TYPE = "dataset"

# Sakhiur's five categories. Swap this list if running for Sadman's categories instead.
CATEGORIES = ["cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE"]

CONTENT_TYPES = ["abstracts", "pdf_chunks"]

# Which (model, content_type) combinations actually need generating.
# Explicit rather than "generate everything" so re-running after a partial failure
# doesn't waste OpenAI budget or GPU time on work that's already done.
GENERATION_PLAN = {
    "text-embedding-3-large": ["pdf_chunks"],          # abstracts already exist, full corpus
    "BAAI/bge-small-en-v1.5": ["abstracts", "pdf_chunks"],
    "Alibaba-NLP/gte-base-en-v1.5": ["abstracts", "pdf_chunks"],
}

# Matches the cap already used when generating text-embedding-3-small's pdf_chunks.
# Applying the same cap here keeps the benchmark comparison fair: every model is
# scored against the same set of chunks, not a bigger or smaller sample.
MAX_PDFS_PER_CATEGORY = 100

BATCH_SIZE = 100              # texts per API call / per encode() batch
REQUEST_DELAY_SECONDS = 0.5   # small delay between OpenAI batches to stay under rate limits
FORCE_REGENERATE = False      # set True to re-embed categories that already have output files

LOCAL_OUTPUT_ROOT = Path("data/embeddings")

## Load source text from Hugging Face

Real schema, confirmed against the actual dataset (not assumed): abstracts records use `id` + `text`; pdf_chunks records use `paper_id` + `page_num` + `text` (plus `category`, `page_type`, `char_count`, `content_hash`, `is_duplicate`, `source_path`, which we don't need for embedding). The pdf_chunks `chunk_id` isn't stored directly, it's constructed as `{paper_id}_p{page_num}`, matching the convention already used by the `text-embedding-3-small` pdf_chunks files, so IDs line up across all models without any remapping.

Downloaded via `hf_hub_download` + manual JSONL parsing (not `datasets.load_dataset`), matching the pattern already working in your capped-small-model notebook. Missing files (e.g. a collaborator's category not yet uploaded) are reported and skipped rather than raising, so one missing file doesn't kill the whole run.

In [ ]:
from huggingface_hub import hf_hub_download


def load_category_records(category: str, content_type: str) -> list[dict]:
    """
    content_type: 'abstracts' or 'pdf_chunks'
    Returns a list of parsed JSON records, or an empty list if the file doesn't
    exist on the Hub yet.
    """
    if content_type == "abstracts":
        repo_path = f"abstracts/by_category/{category}.jsonl"
    elif content_type == "pdf_chunks":
        repo_path = f"pdf_chunks/{category}.jsonl"
    else:
        raise ValueError(f"Unknown content_type: {content_type}")

    try:
        local_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, filename=repo_path)
    except Exception as e:
        print(f"  [SKIP] {repo_path} not found on Hub ({e})")
        return []

    records = []
    with open(local_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  [WARN] {repo_path} line {line_num}: malformed JSON, skipping")
    return records


def get_chunk_id_and_text(record: dict, content_type: str) -> tuple[str, str]:
    """Extracts (chunk_id, text_to_embed) from a record, per content_type's real schema."""
    if content_type == "abstracts":
        return record["id"], record["text"]
    elif content_type == "pdf_chunks":
        chunk_id = f"{record['paper_id']}_p{record['page_num']}"
        return chunk_id, record["text"]
    else:
        raise ValueError(f"Unknown content_type: {content_type}")

## PDF chunk capping (matches the existing text-embedding-3-small run exactly)

Caps by distinct `paper_id`, not by raw chunk/page count: the first `MAX_PDFS_PER_CATEGORY` paper IDs (sorted for determinism, so re-runs always select the same papers rather than an order that depends on file iteration) are kept, and *all* pages belonging to those selected papers are included. This means the actual number of embedded chunks per category will be higher than `MAX_PDFS_PER_CATEGORY`, since each paper contributes multiple pages, that's expected and matches how the existing capped file was built.

In [ ]:
def cap_pdf_records_by_paper(records: list[dict], max_papers: int) -> list[dict]:
    unique_paper_ids = sorted({r["paper_id"] for r in records})
    selected_paper_ids = set(unique_paper_ids[:max_papers])
    filtered = [r for r in records if r["paper_id"] in selected_paper_ids]
    return filtered

## Skip-existing resumability

Generation over the corpus with an open-source model on a single Colab GPU can take a while, and Colab sessions can disconnect. Checking which chunk_ids are already written lets a re-run pick up where it left off instead of restarting from scratch.

In [ ]:
def load_existing_ids(output_path: Path) -> set[str]:
    if not output_path.exists():
        return set()
    existing = set()
    with open(output_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            existing.add(json.loads(line)["chunk_id"])
    return existing


def append_embeddings(output_path: Path, records: list[dict]) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "a") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

## Embedding functions

One function per model family. `text-embedding-3-large` goes through the raw `openai` client (not `langchain-openai`) with basic retry-on-failure, in keeping with the project's preference for pure Python clients over LangChain in the benchmarking pipeline, since LangChain's abstraction adds latency overhead that would pollute anything downstream that cares about timing. Open-source models are batched on GPU (if available) for throughput.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    if IN_COLAB:
        print("No GPU detected. Runtime > Change runtime type > GPU, then re-run this cell.")
    else:
        print(
            "No GPU detected on this machine. Generation will still work on CPU, "
            "just slower for the open-source models. To use a GPU instead, either "
            "run this notebook on a machine with a CUDA-capable GPU + the matching "
            "torch build installed, or run it on Colab (Runtime > Change runtime type > GPU)."
        )

In [ ]:
_st_model_cache: dict[str, SentenceTransformer] = {}


def embed_with_sentence_transformers(
    model_name: str, chunk_ids: list[str], texts: list[str], batch_size: int = BATCH_SIZE
) -> list[dict]:
    if model_name not in _st_model_cache:
        _st_model_cache[model_name] = SentenceTransformer(model_name, device=device)
    model = _st_model_cache[model_name]

    # normalize_embeddings=True makes the vectors unit-length, which is what both
    # BGE and GTE expect for cosine-similarity retrieval and keeps this consistent
    # with how OpenAI embeddings are typically consumed downstream.
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    return [
        {"chunk_id": cid, "embedding": vec.tolist()}
        for cid, vec in zip(chunk_ids, vectors)
    ]

In [ ]:
from openai import OpenAI

openai_client = OpenAI()


def embed_with_openai(
    model_name: str,
    chunk_ids: list[str],
    texts: list[str],
    batch_size: int = BATCH_SIZE,
    max_retries: int = 3,
) -> list[dict]:
    results = []
    for start in range(0, len(texts), batch_size):
        batch_ids = chunk_ids[start:start + batch_size]
        batch_texts = texts[start:start + batch_size]

        response = None
        for attempt in range(1, max_retries + 1):
            try:
                response = openai_client.embeddings.create(model=model_name, input=batch_texts)
                break
            except Exception as e:
                print(f"  [WARN] batch {start}-{start + len(batch_texts)}: attempt {attempt} failed ({e})")
                if attempt == max_retries:
                    print(f"  [ERROR] batch {start}-{start + len(batch_texts)}: giving up after {max_retries} attempts")
                else:
                    time.sleep(2 ** attempt)

        if response is not None:
            for cid, item in zip(batch_ids, response.data):
                results.append({"chunk_id": cid, "embedding": item.embedding})

        time.sleep(REQUEST_DELAY_SECONDS)

    return results


def is_openai_model(model_name: str) -> bool:
    return model_name.startswith("text-embedding-")


def embed_records(model_name: str, chunk_ids: list[str], texts: list[str]) -> list[dict]:
    if is_openai_model(model_name):
        return embed_with_openai(model_name, chunk_ids, texts)
    else:
        return embed_with_sentence_transformers(model_name, chunk_ids, texts)

## Run generation

Loops over the generation plan: for each (model, content_type) pair, for each category, load source records from HF, apply the paper-cap for pdf_chunks only, skip chunk_ids already embedded, embed the rest, append to the jsonl file. Structured this way so a crash partway through only loses the current in-flight batch, not prior progress, and re-running the whole notebook is safe and cheap once most of the work is already done.

In [ ]:
generation_summary = {}

for model_name, content_types in GENERATION_PLAN.items():
    for content_type in content_types:
        print(f"\n{'=' * 70}")
        print(f"  {model_name} | {content_type}")
        print(f"{'=' * 70}")

        for category in CATEGORIES:
            output_path = LOCAL_OUTPUT_ROOT / model_name / content_type / f"{category}.jsonl"

            if output_path.exists() and not FORCE_REGENERATE:
                existing_ids = load_existing_ids(output_path)
            else:
                existing_ids = set()

            all_records = load_category_records(category, content_type)
            if not all_records:
                generation_summary[(model_name, content_type, category)] = 0
                continue

            if content_type == "pdf_chunks":
                total_papers_available = len({r["paper_id"] for r in all_records})
                records = cap_pdf_records_by_paper(all_records, MAX_PDFS_PER_CATEGORY)
                papers_selected = len({r["paper_id"] for r in records})
                print(
                    f"  {category}: {papers_selected}/{total_papers_available} papers selected "
                    f"(cap={MAX_PDFS_PER_CATEGORY}), {len(records)} page-chunks in scope"
                )
            else:
                records = all_records
                print(f"  {category}: {len(records)} abstracts in scope (full corpus)")

            chunk_ids = []
            texts = []
            for record in records:
                cid, text = get_chunk_id_and_text(record, content_type)
                if cid in existing_ids:
                    continue
                if not text or not text.strip():
                    print(f"    [WARN] {cid}: empty text, skipping")
                    continue
                chunk_ids.append(cid)
                texts.append(text)

            if not chunk_ids:
                print(f"  {category}: nothing new to embed, all chunks already present")
                generation_summary[(model_name, content_type, category)] = len(existing_ids)
                continue

            print(f"  {category}: embedding {len(chunk_ids)} new chunks")
            new_embeddings = embed_records(model_name, chunk_ids, texts)
            append_embeddings(output_path, new_embeddings)

            total_now = len(existing_ids) + len(new_embeddings)
            generation_summary[(model_name, content_type, category)] = total_now
            print(f"  {category}: wrote {len(new_embeddings)} embeddings -> {output_path} (total: {total_now})")

## Verify output

Quick sanity check before uploading: confirm every planned (model, content_type, category) combination produced a file with the expected number of records, and that embedding dimensions are consistent within a model. Catching a truncated or malformed file here is much cheaper than discovering it mid-evaluation.

In [ ]:
for model_name, content_types in GENERATION_PLAN.items():
    print(f"\nModel: {model_name}")
    for content_type in content_types:
        print(f"  {content_type}:")
        for category in CATEGORIES:
            output_path = LOCAL_OUTPUT_ROOT / model_name / content_type / f"{category}.jsonl"
            if not output_path.exists():
                print(f"    {category}: MISSING (no output file)")
                continue

            dims = set()
            count = 0
            with open(output_path, "r") as f:
                for line in f:
                    record = json.loads(line)
                    dims.add(len(record["embedding"]))
                    count += 1

            dim_str = dims.pop() if len(dims) == 1 else f"INCONSISTENT: {dims}"
            print(f"    {category}: {count} vectors, dim={dim_str}")

## Upload to Hugging Face

Pushes the newly generated embeddings to the private dataset repo alongside the existing `text-embedding-3-small` files, so the evaluation notebook (run locally or on any machine) can pull everything from one place rather than needing these files copied around manually. Path convention (`embeddings/{model_name}/{content_type}/{category}.jsonl`) matches what's already on the Hub for `text-embedding-3-small`.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

for model_name, content_types in GENERATION_PLAN.items():
    for content_type in content_types:
        for category in CATEGORIES:
            local_path = LOCAL_OUTPUT_ROOT / model_name / content_type / f"{category}.jsonl"
            if not local_path.exists():
                continue

            repo_path = f"embeddings/{model_name}/{content_type}/{category}.jsonl"
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=repo_path,
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                commit_message=f"Upload {model_name} embeddings for {content_type}/{category}.jsonl",
            )
            print(f"uploaded {repo_path}")

## Corpus Composition Summary (for the paper)

Descriptive statistics characterizing the embedded corpus, meant to justify and document the sample used across all four models in the embedding benchmark: how many papers and chunks per category, the page_type composition (TEXT/TABLE/MATH/VISUAL from the DPRA routing), typical chunk length, duplicate rate, and how much text needed truncation to fit the 8192-token embedding limit.

Built by joining what's actually in each model's written embeddings jsonl (the `chunk_id`s that succeeded) against the original HF source records (`paper_id`, `category`, `page_num`, `page_type`, `char_count`, `is_duplicate`), so these numbers reflect the real embedded sample, not the pre-cap corpus.

In [ ]:
import tiktoken
import pandas as pd
_summary_tiktoken_encoding = tiktoken.get_encoding("cl100k_base")


def build_corpus_summary() -> pd.DataFrame:
    """One row per (model, content_type, category), joining embedded chunk_ids
    against HF source record metadata (page_type, char_count, is_duplicate).
    """
    rows = []

    for model_name, content_types in GENERATION_PLAN.items():
        for content_type in content_types:
            for category in CATEGORIES:
                output_path = LOCAL_OUTPUT_ROOT / model_name / content_type / f"{category}.jsonl"
                if not output_path.exists():
                    continue

                embedded_ids = set()
                dims = set()
                with open(output_path, "r") as f:
                    for line in f:
                        record = json.loads(line)
                        embedded_ids.add(record["chunk_id"])
                        dims.add(len(record["embedding"]))

                source_records = load_category_records(category, content_type)

                for record in source_records:
                    cid, text = get_chunk_id_and_text(record, content_type)
                    if cid not in embedded_ids:
                        continue

                    token_count = len(_summary_tiktoken_encoding.encode(text)) if text else 0

                    rows.append({
                        "model": model_name,
                        "content_type": content_type,
                        "category": category,
                        "chunk_id": cid,
                        "paper_id": record.get("paper_id", record.get("id")),
                        "page_type": record.get("page_type", "N/A"),
                        "char_count": record.get("char_count", len(text) if text else 0),
                        "token_count": token_count,
                        "was_truncated": token_count > 8192,
                        "is_duplicate": record.get("is_duplicate", False),
                        "embedding_dim": dims.pop() if len(dims) == 1 else None,
                    })

    return pd.DataFrame(rows)


print("Building corpus summary (re-reading cached HF source files, no re-download)...")
summary_df = build_corpus_summary()
print(f"{len(summary_df)} embedded chunks summarized across {summary_df['model'].nunique()} models")